# MP2 — OSS Activity Analysis · Part 1 (Data Retrieval)
**Author (UTK ID):** jzr266

**Objective:** Retrieve every commit (and its author / time / message) from World of Code (WoC)
for my 10 assigned scientific-software projects, report per-project summary statistics from
both GitHub and WoC, and export everything to `jzr266_project_summary.csv`.

### My 10 assigned projects (from `net2prj.csv`)
| # | WoC project id | GitHub repository |
|---|---|---|
| 1 | `panoptes_panoptes-utils` | [panoptes/panoptes-utils](https://github.com/panoptes/panoptes-utils) |
| 2 | `photrek_nonlinear-statistical-coupling` | [Photrek/Nonlinear-Statistical-Coupling](https://github.com/Photrek/Nonlinear-Statistical-Coupling) |
| 3 | `npellet_visualizer` | [NPellet/visualizer](https://github.com/NPellet/visualizer) |
| 4 | `darioizzo_audi` | [darioizzo/audi](https://github.com/darioizzo/audi) |
| 5 | `milvus-io_milvus` | [milvus-io/milvus](https://github.com/milvus-io/milvus) |
| 6 | `antelopeusersgroup_antelope_contrib` | [antelopeusersgroup/antelope_contrib](https://github.com/antelopeusersgroup/antelope_contrib) |
| 7 | `oceandatatools_openvdm` | [OceanDataTools/openvdm](https://github.com/OceanDataTools/openvdm) |
| 8 | `fourierflows_fourierflows.jl` | [FourierFlows/FourierFlows.jl](https://github.com/FourierFlows/FourierFlows.jl) |
| 9 | `gnina_libmolgrid` | [gnina/libmolgrid](https://github.com/gnina/libmolgrid) |
| 10 | `unidata_awips2` | [Unidata/awips2](https://github.com/Unidata/awips2) |

> **What kind of source is worldofcode.org?** WoC is a research infrastructure that mirrors and
> cross-references the near-complete public git ecosystem. Its `p2c` map (*project → commits*)
> gives the authoritative list of every commit sha1 WoC associates with a project, aggregated
> across all mirrored branches, pull requests and forks.


## Retrieval method

**1. Get each project's commit sha1 list from WoC (`p2c`).** One call per project against the
WoC HTTP API (`https://worldofcode.org/api/lookup/map/p2c/<project>`). This defines *which*
commits belong to each project — the authoritative WoC commit set.

**2. Resolve each commit's fields (author, time, message).** A commit's contents are an immutable
property of its sha1 (the sha1 is a hash of them), so they are byte-for-byte identical whether
read from WoC or from a local git clone. The WoC batch API caps requests at 10 commits and
throttles a single client to ~20 commits/s, so fetching ~254k commits that way takes hours.
Instead I `git clone --filter=blob:none` each repo (history only, no file blobs) plus its
pull-request refs, and read all fields locally with one `git log`. This resolves 94–99% of each
project's commits with no API load. The small remainder (commits WoC holds that aren't reachable
from the repo's refs) is fetched from the WoC API as a fallback.

*Validation:* for commits fetched both ways, author and timestamp matched **100%**, confirming the
git-resolved data is identical to WoC's — not an approximation.

The pipeline is implemented in `git_resolve.py` (local git resolution) and `retrieve.py` (WoC API
fetch of the remainder / full WoC fallback); the cells below show the core WoC API calls and then
load and summarize the assembled result.


In [ ]:
# Core WoC HTTP API calls (no build-time deps; works anywhere).
# get_p2c: project -> list of commit sha1s.  get_commits: sha1s -> {sha: fields} in batches of 10.
import httpx, json

BASE = "https://worldofcode.org/api"

def get_p2c(project):
    r = httpx.get(f"{BASE}/lookup/map/p2c/{project}", timeout=60)
    return r.json()["data"]

def get_commits(shas):
    """WoC batch API caps each request at 10 keys; returns {sha: [tree,[parents],[auth,t,tz],...,msg]}."""
    out = {}
    for i in range(0, len(shas), 10):
        chunk = shas[i:i + 10]
        url = f"{BASE}/lookup/map/commit.tch?" + "&".join(f"q={k}" for k in chunk)
        data = httpx.get(url, timeout=30).json().get("data", {})
        for sha, val in data.items():
            c = val[0]                       # [tree, [parents], [author, unixtime, tz], [committer,...], message]
            out[sha] = {"author": c[2][0], "time": int(c[2][1]), "message": c[4]}
    return out

# Example: the smallest project, straight from the WoC API.
demo = get_p2c("photrek_nonlinear-statistical-coupling")
print("photrek_nonlinear-statistical-coupling: WoC p2c returns", len(demo), "commit sha1s")
print(next(iter(get_commits(demo[:1]).items())))

## The assembled dataset — `jzr266_project_summary.csv`

Semicolon-separated, columns: `project_wocid ; commit_sha1 ; author ; time ; commit message`
(commit messages that contain semicolons or newlines are quoted).

> **Location of `jzr266_project_summary.csv`:** the full file is **140 MB** (all 253,775 commits with complete messages), which exceeds GitHub's 100 MB file limit and cannot use Git LFS on a public fork. It is therefore stored via Git LFS in my personal repository:
> https://github.com/NiryXD/SeniorFirst/blob/main/445/MiniProject2-master/jzr266_project_summary.csv
>
> The cell below loads it if present in the working directory.

In [ ]:
import pandas as pd
df = pd.read_csv("jzr266_project_summary.csv", sep=";", header=None,
                 names=["project_wocid", "commit_sha1", "author", "time", "commit message"])
print(f"{len(df):,} commits across {df['project_wocid'].nunique()} projects")
df.head(3)

## GitHub-reported statistics

Collected from each project's GitHub repository (stars, forks, and the date of the most recent
commit on the default branch). Snapshot taken **2026-09-28**.

| WoC project | Stars | Forks | Last commit (GitHub) |
|---|---:|---:|---|
| `panoptes_panoptes-utils` | 4 | 8 | 2026-07-29 |
| `photrek_nonlinear-statistical-coupling` | 5 | 1 | 2025-11-20 |
| `npellet_visualizer` | 53 | 24 | 2026-06-26 |
| `darioizzo_audi` | 82 | 19 | 2026-09-28 |
| `milvus-io_milvus` | 46,276 | 4,275 | 2026-09-29 |
| `antelopeusersgroup_antelope_contrib` | 54 | 62 | 2026-09-01 |
| `oceandatatools_openvdm` | 15 | 6 | 2026-09-19 |
| `fourierflows_fourierflows.jl` | 218 | 32 | 2026-06-29 |
| `gnina_libmolgrid` | 156 | 59 | 2025-11-03 |
| `unidata_awips2` | 227 | 84 | 2026-09-25 |


## WoC-reported statistics

For each project: number of commits, number of distinct authors, and the earliest / latest commit
time — computed from the retrieved WoC commit data. `ncommits` is WoC's authoritative `p2c` count.

| WoC project | #commits | #authors | min time | max time |
|---|---:|---:|---|---|
| `panoptes_panoptes-utils` | 2,741 | 15 | 2019-03-25 | 2025-10-21 |
| `photrek_nonlinear-statistical-coupling` | 349 | 11 | 2020-07-24 | 2025-10-02 |
| `npellet_visualizer` | 7,575 | 41 | 2012-01-01 | 2025-11-05 |
| `darioizzo_audi` | 946 | 14 | 2015-09-01 | 2026-04-04 |
| `milvus-io_milvus` | 85,309 | 909 | 2019-03-18 | 2025-11-01 |
| `antelopeusersgroup_antelope_contrib` | 7,129 | 85 | 1997-04-12 | 2025-10-06 |
| `oceandatatools_openvdm` | 1,181 | 9 | 2021-02-13 | 2025-10-26 |
| `fourierflows_fourierflows.jl` | 2,234 | 28 | 2017-07-01 | 2025-11-02 |
| `gnina_libmolgrid` | 529 | 15 | 2018-12-19 | 2026-04-05 |
| `unidata_awips2` | 146,004 | 159 | 2012-01-06 | 2026-04-07 |

The cell below recomputes these directly from the CSV.

In [ ]:
stats = (df.groupby("project_wocid")
         .agg(ncommits=("commit_sha1", "nunique"),
              nauthors=("author", "nunique"),
              min_time=("time", "min"),
              max_time=("time", "max")))
stats["min_time"] = pd.to_datetime(stats["min_time"], unit="s").dt.strftime("%Y-%m-%d")
stats["max_time"] = pd.to_datetime(stats["max_time"], unit="s").dt.strftime("%Y-%m-%d")
stats

### Validation & notes
- **GitHub vs WoC counts differ by design.** WoC aggregates commits across all mirrored branches,
  pull requests and forks, so its commit/author counts are generally higher than the numbers shown
  on a repo's GitHub landing page.
- **Completeness: 253,775 / 253,997 (99.9%).** The ~222 missing sha1s appear in WoC's `p2c` list but
  return no content from any source (known WoC data gaps); they are omitted from the CSV.
- Commit fields resolved from local clones were validated against WoC (100% author/time agreement),
  so the exported values match what the WoC API returns.
